# Do the seed budgets cut rankings on plateaus?

**Hypothesis.** Deterministic selectors break score ties by actor id, and MLTM itself is deterministic,
so the 30 repetitions of a deterministic selector are the same run. If a budget cut-off falls inside a
block of equally scored actors (a plateau), the seed set is an arbitrary, id-driven pick out of many
equivalent ones, and the reported result is a single sample whose spread we never see.
If cut-offs fall on steep parts of the sorted score curve, the seed set is well defined.

**Method.** For every network, selector (`data/rankings/v2`) and budget `b`, the cut-off is
`k = max(1, round(b * N))` (as in `sqs.pipeline.spreading`). At `k` we find the tie block, i.e. the
contiguous run of actors with the same score key as the `k`-th one, and report:

* `block`: size of the tie block, `inside`: how many of them are seeded, `left_out = block - inside`;
* `arbitrary`: `inside / k`, the share of the seed set picked by actor id rather than by score;
* `log10_sets`: `log10 C(block, inside)`, how many equally good seed sets exist.

The cut-off is safe iff `left_out == 0` (the block ends exactly at `k`).

Static SQ / SQ_R skip actors that are already active, so they read the ranking past `k`; `k` is a lower
bound. Because of this we also report the largest tie block within `[k, k + band]` (`band` = `BAND` · N).

`mds+deg` is ranked per repetition, so it is evaluated per rep, plus the mean pairwise Jaccard
similarity of its top-`k` sets across reps (how much variability the reps really have).

Run the notebook top to bottom; it writes the PDF to `OUT_PDF`.

In [ ]:
import json
from itertools import combinations
from math import comb, log10
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RANKINGS = ROOT / "data/rankings/v2"
OUT_PDF = ROOT / "scripts/out/ranking_plateaus.pdf"

# configs/grids/static.yaml (configs/grids/strategies.yaml uses subsets of these)
BUDGETS = {"OR": [0.05, 0.1, 0.15, 0.2, 0.25], "AND": [0.15, 0.2, 0.25, 0.3, 0.35]}
ALL_BUDGETS = sorted({b for bs in BUDGETS.values() for b in bs})
SELECTORS = ["deg-c-d", "nghb-sd", "voterank", "ts-net", "mds+deg"]
BAND = 0.05  # fraction of N past the cut-off that SQ / SQ_R may still read
SIG_DIGITS = 6  # float scores equal up to this many significant digits count as tied
# ts-net order (sqs.runners.selectors.ts_net._SORT_SPEC): name, ascending
TS_NET_SPEC = [
    ("exposed", False),
    ("simulation_length", True),
    ("peak_infected", True),
    ("peak_iteration", False),
]

NETWORKS = sorted(p.name for p in RANKINGS.iterdir() if p.is_dir())
pd.set_option("display.width", 200, "display.max_rows", 400)
plt.rcParams.update({"figure.dpi": 110, "font.size": 8})
print(NETWORKS)

## Loading

Each ranking becomes a frame with the plotted `value` and a hashable tie `key`:

* `deg-c-d`, `nghb-sd`: discounted score when picked; `voterank`: votes when elected (0 for the unelected tail);
* `ts-net`: the lexicographic key of the four predicted outputs, plotted by `exposed`;
* `mds+deg`: `(driver, degree)`, plotted by degree (drivers come first).

In [ ]:
def _round(x):
    if x is None:
        return None
    x = float(x)
    return float(f"{x:.{SIG_DIGITS}g}")


def _key_value(selector, score):
    if score is None:
        return None, np.nan
    if selector == "ts-net":
        return tuple(_round(score[n]) for n, _ in TS_NET_SPEC), score["exposed"]
    if selector == "mds+deg":
        return (score["driver"], score.get("degree")), score.get("degree", np.nan)
    return _round(score), float(score)


def load(network, selector, rep=None):
    """Load one v2 ranking as a frame with actor, plotted value and tie key."""
    if selector == "ts-net":
        files = sorted((RANKINGS / network).glob("ts-net__*.json"))
        assert len(files) == 1, files
        path = files[0]
    elif selector == "mds+deg":
        path = RANKINGS / network / f"mds+deg__rep{rep}.json"
    else:
        path = RANKINGS / network / f"{selector}.json"
    entries = json.loads(path.read_text())
    keys, values = zip(*(_key_value(selector, e["score"]) for e in entries))
    frame = pd.DataFrame({"actor": [e["actor"] for e in entries], "value": values})
    frame["key"] = list(keys)
    if selector == "mds+deg":
        frame["driver"] = [e["score"]["driver"] for e in entries]
    return frame


def mds_reps(network):
    """Return the repetitions with an mds+deg ranking for ``network``."""
    return sorted(
        int(p.stem.split("rep")[1]) for p in (RANKINGS / network).glob("mds+deg__rep*.json")
    )


rankings = {}  # (network, selector) -> frame, or list of frames for mds+deg
for net in NETWORKS:
    for sel in SELECTORS:
        rankings[net, sel] = (
            [load(net, sel, r) for r in mds_reps(net)] if sel == "mds+deg" else load(net, sel)
        )
sizes = {net: len(rankings[net, "deg-c-d"]) for net in NETWORKS}
sizes

## Tie blocks at the cut-offs

In [ ]:
def blocks(keys):
    """Return (start, end) of the run of equal keys containing each position."""
    ids = np.zeros(len(keys), dtype=int)
    for i in range(1, len(keys)):
        ids[i] = ids[i - 1] + (keys[i] != keys[i - 1] or keys[i] is None)
    starts = pd.Series(np.arange(len(keys))).groupby(ids).transform("min").to_numpy()
    ends = pd.Series(np.arange(len(keys))).groupby(ids).transform("max").to_numpy() + 1
    return starts, ends


def cut_stats(frame, k, band):
    """Describe the tie block at cut-off ``k`` and the largest one ``band`` ranks past it."""
    starts, ends = blocks(list(frame["key"]))
    s, e = starts[k - 1], ends[k - 1]
    block, inside = e - s, k - s
    hi = min(len(frame), k + band)
    band_block = int((ends[k:hi] - starts[k:hi]).max()) if hi > k else 0
    return {
        "k": k,
        "block": block,
        "inside": inside,
        "left_out": block - inside,
        "arbitrary": inside / k if block > inside else 0.0,
        "log10_sets": log10(comb(block, inside)),
        "band_max_block": band_block,
        "safe": block == inside,
    }


rows = []
for (net, sel), ranking in rankings.items():
    n = sizes[net]
    for b in ALL_BUDGETS:
        k = max(1, round(b * n))
        protocols = "+".join(p for p, bs in BUDGETS.items() if b in bs)
        frames = ranking if isinstance(ranking, list) else [ranking]
        for rep, frame in enumerate(frames):
            rows.append(
                {
                    "network": net,
                    "selector": sel,
                    "budget": b,
                    "protocols": protocols,
                    "rep": rep if sel == "mds+deg" else None,
                    **cut_stats(frame, k, max(1, round(BAND * n))),
                }
            )
cuts = pd.DataFrame(rows)

# one row per (network, selector, budget); mds+deg aggregated over reps (worst case + mean)
agg = cuts.groupby(["network", "selector", "budget", "protocols", "k"], as_index=False).agg(
    block=("block", "max"),
    left_out=("left_out", "max"),
    arbitrary=("arbitrary", "max"),
    arbitrary_mean=("arbitrary", "mean"),
    log10_sets=("log10_sets", "max"),
    band_max_block=("band_max_block", "max"),
    safe_share=("safe", "mean"),
)
agg["safe"] = agg["safe_share"] == 1.0
agg

## Across-rep variability of `mds+deg`

Mean pairwise Jaccard similarity of the top-`k` sets of the 30 reps (1.0 = the reps seed the same set).

In [ ]:
def mean_jaccard(frames, k):
    """Return the mean pairwise Jaccard similarity of the top-``k`` sets of ``frames``."""
    sets = [set(f["actor"].iloc[:k]) for f in frames]
    return float(np.mean([len(a & b) / len(a | b) for a, b in combinations(sets, 2)]))


jacc = pd.DataFrame(
    [
        {
            "network": net,
            "budget": b,
            "jaccard": mean_jaccard(rankings[net, "mds+deg"], max(1, round(b * sizes[net]))),
        }
        for net in NETWORKS
        for b in ALL_BUDGETS
    ]
).pivot(index="network", columns="budget", values="jaccard")
jacc.round(3)

## Figures

In [ ]:
BUDGET_COLORS = {"OR": "#1f77b4", "AND": "#d62728", "OR+AND": "#7b3fa0"}


def protocols_of(b):
    """Return the protocols that use budget ``b`` (e.g. 'OR+AND')."""
    return "+".join(p for p, bs in BUDGETS.items() if b in bs)


def plot_ranking(ax, net, sel):
    """Plot the sorted scores of one ranking with its budget cut-offs."""
    n = sizes[net]
    ranking = rankings[net, sel]
    frames = ranking if isinstance(ranking, list) else [ranking]
    x = np.arange(1, n + 1) / n
    for i, frame in enumerate(frames):
        ax.plot(
            x,
            frame["value"],
            lw=0.6 if len(frames) > 1 else 1.0,
            color="0.2",
            alpha=0.25 if len(frames) > 1 else 1.0,
        )
    if sel == "mds+deg":
        drivers = [f["driver"].sum() / n for f in frames]
        ax.axvspan(
            min(drivers),
            max(drivers),
            color="0.85",
            lw=0,
            zorder=0,
            label="end of drivers (range over reps)",
        )
    frame = frames[0]
    starts, ends = blocks(list(frame["key"]))
    for b in ALL_BUDGETS:
        k = max(1, round(b * n))
        color = BUDGET_COLORS[protocols_of(b)]
        s, e = starts[k - 1], ends[k - 1]
        if e - s > k - s:  # straddled tie block
            ax.axvspan(s / n, e / n, color=color, alpha=0.18, lw=0)
        ax.axvline(k / n, color=color, lw=0.8, ls="-" if e - s == k - s else "--")
    ax.set_xlim(0, min(1.0, max(ALL_BUDGETS) + 2 * BAND))
    ax.set_title(sel)
    ax.set_xlabel("rank / N")
    ax.grid(alpha=0.3, lw=0.4)


def network_figure(net):
    """Return the figure with the rankings of all selectors on ``net``."""
    fig, axes = plt.subplots(1, len(SELECTORS), figsize=(16, 3.4))
    for ax, sel in zip(axes, SELECTORS):
        plot_ranking(ax, net, sel)
    axes[0].set_ylabel("score")
    handles = [
        plt.Line2D([], [], color=c, label=f"budget used by {p}") for p, c in BUDGET_COLORS.items()
    ]
    handles += [
        plt.Line2D([], [], color="0.3", ls="--", label="cut-off inside a tie block (shaded)")
    ]
    fig.legend(
        handles=handles, loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.06)
    )
    fig.suptitle(f"{net} (N = {sizes[net]}): sorted scores with budget cut-offs", y=1.02)
    fig.tight_layout()
    return fig


def heatmap_figure(metric, title, fmt, vmax=None):
    """Return one heatmap of ``metric`` per selector (networks x budgets)."""
    fig, axes = plt.subplots(1, len(SELECTORS), figsize=(16, 4.2), sharey=True)
    for ax, sel in zip(axes, SELECTORS):
        table = agg[agg.selector == sel].pivot(index="network", columns="budget", values=metric)
        block = agg[agg.selector == sel].pivot(index="network", columns="budget", values="block")
        im = ax.imshow(
            table.to_numpy(),
            cmap="Reds",
            vmin=0,
            vmax=vmax or max(1e-9, agg[metric].max()),
            aspect="auto",
        )
        for (i, j), v in np.ndenumerate(table.to_numpy()):
            ax.text(j, i, fmt(v, block.to_numpy()[i, j]), ha="center", va="center", fontsize=6)
        ax.set_xticks(range(len(table.columns)), [f"{b:g}" for b in table.columns])
        ax.set_yticks(range(len(table.index)), table.index)
        ax.set_title(sel)
        ax.set_xlabel("budget")
    fig.colorbar(im, ax=axes, shrink=0.8, label=metric)
    fig.suptitle(title)
    return fig


FIGURES = []

CAPTIONS = {
    "arbitrary": """
        What it shows: each cell is one experimental setting (network x budget). The seeds
        are the top k actors of the ranking (k = budget x N). If the k-th actor has the same
        score as some actors just below it, the code cannot tell them apart and simply picks
        them in order of their ID. The top number is the share of seeds chosen this way; the
        bottom number (/m) is how many actors share that score.

        GOOD for us: an empty or pale cell. The seed set is decided by the scores alone.

        BAD for us: a dark cell. A large part of the seed set comes down to ID order, so the
        result is one arbitrary draw, and the 30 repetitions do not reveal it.

        Example: '0.26 /21' means 21 actors share a score, 13 of them get in and 8 do not;
        those 13 make up 26% of the 50 seeds. For mds+deg the cell shows the worst of the 30
        repetitions.
    """,
    "log10_sets": """
        What it shows: the same settings as the previous page, but counting how many
        different seed sets the scores would justify equally well. The cell shows the number
        of digits of that count (log10): 1 means about 10 sets, 5.3 about 200,000 sets.

        GOOD for us: an empty cell. There is only one correct seed set.

        BAD for us: a large number. Many seed sets are equally valid, yet the experiments
        only ever try one of them.

        This page tells the same story as the previous one; it simply makes the extreme cases
        (e.g. arxiv) easier to spot.
    """,
    "band_max_block": f"""
        What it shows: in the sequential modes (SQ, SQ_R), actors that are already active are
        skipped, so a run reaches further down the ranking than the budget line suggests. This
        page looks at the next {BAND:.0%} of ranks after the budget line and reports the
        largest group of actors sharing one score there.

        GOOD for us: an empty cell or a small number. Even if a run reads further, its choices
        are still driven by the scores.

        BAD for us: a large number. A run that reads that far will again choose among many
        identically scored actors by ID.

        Treat it as a warning sign rather than a precise measurement: we do not know exactly
        how far each run reads.
    """,
    "jaccard": """
        What it shows: mds+deg is the only method whose ranking is recomputed for every
        repetition (its search is randomised). For each setting we compare the seed sets of
        the 30 repetitions pair by pair and report their average overlap (1 = identical
        seeds, 0.5 = about half shared).

        GOOD for us: values clearly below 1 (blue-green). The repetitions really use
        different seeds, so the spread of the results reflects genuine variability.

        BAD for us: values close to 1 (yellow). The repetitions use (almost) the same seeds
        and add little information, just like the other four methods, which always score
        exactly 1.

        Note: arxiv scores 1.00 everywhere, so on this network mds+deg does not vary between
        repetitions either.
    """,
    "network": """
        What it shows: actors sorted from best (left) to worst (right) by the score each
        method gives them. The vertical lines are the budgets used in the experiments (blue:
        OR only, red: AND only, purple: both).

        GOOD for us: a solid line where the curve is steep. Actors just inside and just
        outside the budget have clearly different scores, so the seed set is well defined.

        BAD for us: a dashed line inside a shaded band, where the curve is flat. Every actor
        in the band has the same score and the budget splits them, so who becomes a seed is
        decided by ID.

        mds+deg: one faint curve per repetition (darker where they agree). Drivers come first
        and the remaining actors second, each group sorted by degree, which is why the curve
        jumps back up where the drivers end (grey band). Its budget lines and shading are for
        repetition 0 only.

        Scores: deg-c-d and nghb-sd: discounted score when chosen; voterank: votes when
        elected; ts-net: predicted 'exposed'.
    """,
}


def caption(fig, text, y=-0.02):
    """Put ``text`` under ``fig``, one wrapped paragraph per blank-line-separated block."""
    import textwrap  # noqa: PLC0415

    width = int(fig.get_figwidth() * 15)
    pars = textwrap.dedent(text).strip().split("\n\n")
    lines = [textwrap.fill(" ".join(par.split()), width) for par in pars]
    fig.text(0.01, y, "\n".join(lines), ha="left", va="top", fontsize=8, linespacing=1.4)
    return fig

In [ ]:
fig = heatmap_figure(
    "arbitrary",
    "Share of the seed set picked by actor id at the cut-off "
    "(cell: share / tie block size; mds+deg: worst rep)",
    lambda v, blk: "" if v == 0 else f"{v:.2f}\n/{int(blk)}",
    vmax=1.0,
)
FIGURES.append(caption(fig, CAPTIONS["arbitrary"]))

In [ ]:
fig = heatmap_figure(
    "log10_sets",
    "log10 of the number of equally good seed sets at the cut-off (0 = unique)",
    lambda v, blk: "" if v == 0 else f"{v:.1f}",
)
FIGURES.append(caption(fig, CAPTIONS["log10_sets"]))

In [ ]:
fig = heatmap_figure(
    "band_max_block",
    f"Largest tie block within {BAND:g}·N past the cut-off (what SQ / SQ_R may read further on)",
    lambda v, blk: "" if v <= 1 else f"{int(v)}",
)
FIGURES.append(caption(fig, CAPTIONS["band_max_block"]))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
im = ax.imshow(jacc.to_numpy(), cmap="viridis", vmin=0, vmax=1, aspect="auto")
for (i, j), v in np.ndenumerate(jacc.to_numpy()):
    ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=6, color="w" if v < 0.6 else "k")
ax.set_xticks(range(len(jacc.columns)), [f"{b:g}" for b in jacc.columns])
ax.set_yticks(range(len(jacc.index)), jacc.index)
ax.set_xlabel("budget")
fig.colorbar(im, ax=ax, label="mean pairwise Jaccard of top-k")
ax.set_title("mds+deg: similarity of seed sets across the 30 reps")
fig.tight_layout()
FIGURES.append(caption(fig, CAPTIONS["jaccard"], y=-0.04))

In [ ]:
for net in NETWORKS:
    FIGURES.append(caption(network_figure(net), CAPTIONS["network"], y=-0.13))

## Verdict

In [ ]:
unsafe = agg[~agg.safe]
summary = (
    agg.assign(unsafe=~agg.safe)
    .groupby("selector")
    .agg(
        cells=("unsafe", "size"),
        unsafe=("unsafe", "sum"),
        worst_arbitrary=("arbitrary", "max"),
        mean_arbitrary=("arbitrary", "mean"),
    )
    .reindex(SELECTORS)
)
lines = [
    f"Cut-offs checked: {len(agg)} ({len(NETWORKS)} networks x {len(SELECTORS)} selectors "
    f"x {len(ALL_BUDGETS)} budgets).",
    f"Cut-offs inside a tie block: {len(unsafe)} ({len(unsafe) / len(agg):.0%}).",
    "",
    "Per selector (arbitrary = share of seeds picked by actor id; mds+deg = worst rep):",
]
for sel, r in summary.iterrows():
    lines.append(
        f"  {sel:9s} unsafe {int(r.unsafe):3d}/{int(r.cells)}  "
        f"worst {r.worst_arbitrary:.2f}  mean {r.mean_arbitrary:.3f}"
    )
lines += ["", "Worst cut-offs:"]
for _, r in agg.sort_values("arbitrary", ascending=False).head(15).iterrows():
    if r.arbitrary == 0:
        break
    lines.append(
        f"  {r.network:30s} {r.selector:9s} b={r.budget:<5g} k={r.k:<6d} block={r.block:<6d} "
        f"left_out={r.left_out:<6d} arbitrary={r.arbitrary:.2f}  log10(sets)={r.log10_sets:.1f}"
    )
overlap = jacc.to_numpy()
lines += [
    "",
    "mds+deg mean pairwise Jaccard of top-k across reps: "
    f"min {overlap.min():.2f}, median {np.median(overlap):.2f}, max {overlap.max():.2f}.",
]
VERDICT = "\n".join(lines)
print(VERDICT)

In [ ]:
def text_page(title, body):
    """Return a figure with a title and monospace text, used as a PDF page."""
    fig = plt.figure(figsize=(11.7, 8.3))
    fig.text(0.05, 0.95, title, fontsize=14, weight="bold", va="top")
    fig.text(0.05, 0.90, body, fontsize=8, family="monospace", va="top")
    return fig


INTRO = (
    "Hypothesis: deterministic selectors break ties by actor id and MLTM is deterministic,\n"
    "so all reps of a deterministic selector are the same run. A budget cut-off inside a\n"
    "block of equally scored actors makes the seed set an id-driven pick among many\n"
    "equivalent ones (variance we never observe).\n\n"
    "Cut-off k = max(1, round(b * N)). Tie block = contiguous run of actors whose score key\n"
    f"equals the k-th one (floats compared to {SIG_DIGITS} significant digits; ts-net: full\n"
    "lexicographic key; mds+deg: (driver, degree), evaluated per rep).\n"
    "  arbitrary  = seeds inside the block / k   (0 -> the block ends at k -> safe)\n"
    "  log10_sets = log10 C(block, inside)       (number of equally good seed sets)\n"
    f"  band       = largest block within {BAND:g}*N past k (Static SQ / SQ_R skip active\n"
    "               actors and read deeper than k, so k is only a lower bound of the real\n"
    "               cut-off)\n\n"
    f"Budgets: OR {BUDGETS['OR']}, AND {BUDGETS['AND']}.\n"
    f"Rankings: {RANKINGS.relative_to(ROOT)}\n\n"
)
OUT_PDF.parent.mkdir(parents=True, exist_ok=True)
with PdfPages(OUT_PDF) as pdf:
    pdf.savefig(
        text_page("Do the seed budgets cut rankings on plateaus?", INTRO + VERDICT),
        bbox_inches="tight",
    )
    for fig in FIGURES:
        pdf.savefig(fig, bbox_inches="tight")
    table = unsafe.sort_values(["selector", "network", "budget"])[
        [
            "network",
            "selector",
            "budget",
            "protocols",
            "k",
            "block",
            "left_out",
            "arbitrary",
            "log10_sets",
            "band_max_block",
        ]
    ]
    per_page = 60
    for i in range(0, max(1, len(table)), per_page):
        chunk = table.iloc[i : i + per_page].to_string(
            index=False, float_format=lambda v: f"{v:.3g}"
        )
        pdf.savefig(
            text_page(
                f"Unsafe cut-offs ({i + 1}-{min(i + per_page, len(table))} of {len(table)})", chunk
            ),
            bbox_inches="tight",
        )
print(f"written {OUT_PDF}")